# 03. Анализ распределения доходностей BTC

Цель этого ноутбука — исследовать статистические свойства распределения доходностей Bitcoin и сформулировать выводы, которые будут использованы при построении ML-модели.

**Структура:**
1. Распределение логарифмических доходностей и моменты
2. Тесты на нормальность (Жарка-Бера, Шапиро-Уилка, Колмогорова-Смирнова)
3. Подбор тяжелохвостых распределений (Стьюдент, Lévy α-stable)
4. Анализ стационарности (ADF, KPSS)
5. Автокорреляция и кластеризация волатильности
6. Экспонента Хёрста (R/S-анализ)
7. Выводы для моделирования

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})
np.random.seed(42)

In [ ]:
btc = pd.read_csv("../data/stage/btc.csv", parse_dates=["date"]).set_index("date")
close = btc["close"]

# Логарифмические доходности: r_t = ln(P_t / P_{t-1})
# Аддитивны по времени: ln(P_T/P_0) = sum(r_t), что удобно для статистического анализа
log_ret = np.log(close / close.shift(1)).dropna()

print(f"Период: {log_ret.index.min().date()} — {log_ret.index.max().date()}")
print(f"Наблюдений: {len(log_ret)}")
log_ret.head()

## 1. Распределение логарифмических доходностей

Для финансового анализа используются **логарифмические доходности** $r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$, а не простые ($P_t/P_{t-1} - 1$), по двум причинам:

1. **Аддитивность:** $\ln\!\left(\frac{P_T}{P_0}\right) = \sum_{t=1}^{T} r_t$, что позволяет агрегировать доходности по времени.
2. **Симметричность:** рост и падение на одинаковую логарифмическую величину дают одинаковые абсолютные значения.

Для описания формы распределения вычисляем первые четыре момента:
- **Среднее** $\mu = \frac{1}{n}\sum r_t$
- **Дисперсия** $\sigma^2 = \frac{1}{n-1}\sum (r_t - \mu)^2$
- **Асимметрия (skewness):** $S = \frac{1}{n}\sum\!\left(\frac{r_t - \mu}{\sigma}\right)^3$ — мера несимметричности. $S < 0$ означает, что левый хвост длиннее.
- **Эксцесс (excess kurtosis):** $K = \frac{1}{n}\sum\!\left(\frac{r_t - \mu}{\sigma}\right)^4 - 3$ — мера тяжести хвостов. $K > 0$ (лептокуртоз) означает, что экстремальные значения встречаются чаще, чем у нормального распределения.

In [ ]:
# Первые четыре момента
mu    = log_ret.mean()
sigma = log_ret.std()
skew  = log_ret.skew()
kurt  = log_ret.kurtosis()       # pandas вычисляет excess kurtosis (K-3)

moments = pd.DataFrame({
    "Момент": ["Среднее (μ)", "Ст. откл. (σ)", "Асимметрия (S)", "Эксцесс (K)"],
    "Значение": [mu, sigma, skew, kurt],
    "Нормальное распр.": ["-", "-", "0", "0"],
})
display(moments)

print(f"\nСреднее дневное: {mu:.6f} ({mu*365*100:.2f}% годовых)")
print(f"Дневная волатильность: {sigma:.4f} ({sigma*np.sqrt(365)*100:.1f}% годовых)")

In [ ]:
# Гистограмма логарифмических доходностей с наложенной плотностью N(μ, σ²)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма
x_grid = np.linspace(log_ret.min(), log_ret.max(), 500)
axes[0].hist(log_ret, bins=100, density=True, alpha=0.6, label="Эмпирическое")
axes[0].plot(x_grid, st.norm.pdf(x_grid, mu, sigma), "r-", lw=2, label=f"N({mu:.4f}, {sigma:.4f}²)")
axes[0].set_title("Распределение дневных лог-доходностей BTC")
axes[0].set_xlabel("Лог-доходность")
axes[0].set_ylabel("Плотность")
axes[0].legend()

# QQ-plot против нормального распределения
st.probplot(log_ret, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot: лог-доходности vs N(0,1)")
axes[1].get_lines()[0].set_markersize(2)

plt.tight_layout()
plt.show()

## 2. Тесты на нормальность

Проверяем гипотезу $H_0$: доходности распределены нормально.

| Тест | Что проверяет | Статистика |
|------|---------------|------------|
| **Жарка-Бера** | Совместно S=0 и K=0 | $JB = \frac{n}{6}\left(S^2 + \frac{K^2}{4}\right) \sim \chi^2(2)$ |
| **Шапиро-Уилка** | Отклонение от нормальности (мощный для малых n) | Основан на order statistics |
| **Колмогорова-Смирнова** | $\max|F_{emp}(x) - F_{теор}(x)|$ | $D = \sup_x |F_n(x) - F(x)|$ |

Если $p < 0.05$, отвергаем $H_0$ на уровне значимости 5%.

In [ ]:
# --- Тест Жарка-Бера ---
jb_stat, jb_p = st.jarque_bera(log_ret)

# --- Тест Шапиро-Уилка (на подвыборке 5000, т.к. ограничен по n) ---
sample = log_ret.sample(min(5000, len(log_ret)), random_state=42)
sw_stat, sw_p = st.shapiro(sample)

# --- Тест Колмогорова-Смирнова ---
ks_stat, ks_p = st.kstest(log_ret, "norm", args=(mu, sigma))

alpha = 0.05
results = pd.DataFrame({
    "Тест": ["Жарка-Бера", "Шапиро-Уилка", "Колмогорова-Смирнова"],
    "Статистика": [jb_stat, sw_stat, ks_stat],
    "p-value": [jb_p, sw_p, ks_p],
    "H₀ отвергнута (α=0.05)": [jb_p < alpha, sw_p < alpha, ks_p < alpha],
})
display(results)

if all(results["H₀ отвергнута (α=0.05)"]):
    print("\n>>> Все три теста отвергают нормальность. "
          "Доходности BTC НЕ распределены нормально.")

## 3. Подбор тяжелохвостых распределений

Поскольку нормальность отвергнута, подбираем распределения с более тяжёлыми хвостами.

### Распределение Стьюдента
Плотность: $f(x; \nu) = \frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\sqrt{\nu\pi}\,\Gamma\!\left(\frac{\nu}{2}\right)}\left(1 + \frac{x^2}{\nu}\right)^{-\frac{\nu+1}{2}}$

Параметр $\nu$ (число степеней свободы) определяет тяжесть хвостов: чем меньше $\nu$, тем тяжелее хвосты. При $\nu \to \infty$ распределение сходится к $N(0,1)$.

### Устойчивое распределение Леви (α-stable)
Определяется через характеристическую функцию:
$$\varphi(t) = \exp\!\left(i\mu t - |ct|^\alpha \left[1 - i\beta\,\text{sgn}(t)\tan\!\frac{\pi\alpha}{2}\right]\right)$$

- $\alpha \in (0, 2]$ — индекс устойчивости. При $\alpha = 2$ это гауссиан. При $\alpha < 2$ хвосты убывают как $P(|X|>x) \sim x^{-\alpha}$.
- $\beta \in [-1, 1]$ — параметр асимметрии.
- Если $\alpha < 2$, **дисперсия бесконечна**, и стандартные метрики риска (Sharpe ratio) теоретически не определены.

### Сравнение моделей: AIC
$\text{AIC} = 2k - 2\ln L$, где $k$ — число параметров, $L$ — максимум функции правдоподобия. Меньший AIC — лучшая модель.

In [ ]:
# --- Подбор распределений методом MLE ---
r = log_ret.values

# 1. Нормальное распределение
norm_params = st.norm.fit(r)
norm_ll = st.norm.logpdf(r, *norm_params).sum()

# 2. Распределение Стьюдента (3 параметра: df, loc, scale)
t_params = st.t.fit(r)
t_ll = st.t.logpdf(r, *t_params).sum()

# 3. α-stable распределение (4 параметра: alpha, beta, loc, scale)
stable_params = st.levy_stable.fit(r)
stable_ll = st.levy_stable.logpdf(r, *stable_params).sum()

n = len(r)
aic = lambda ll, k: 2*k - 2*ll

fit_results = pd.DataFrame({
    "Распределение": ["Нормальное N(μ,σ²)", "Стьюдента t(ν,μ,σ)", "Lévy α-stable"],
    "Параметры": [
        f"μ={norm_params[0]:.5f}, σ={norm_params[1]:.5f}",
        f"ν={t_params[0]:.2f}, μ={t_params[1]:.5f}, σ={t_params[2]:.5f}",
        f"α={stable_params[0]:.3f}, β={stable_params[1]:.3f}, μ={stable_params[2]:.5f}, c={stable_params[3]:.5f}",
    ],
    "Log-Likelihood": [norm_ll, t_ll, stable_ll],
    "AIC": [aic(norm_ll, 2), aic(t_ll, 3), aic(stable_ll, 4)],
})
display(fit_results)

print(f"\nЧисло степеней свободы Стьюдента: ν = {t_params[0]:.2f}")
print(f"Индекс устойчивости α-stable: α = {stable_params[0]:.3f}")
if stable_params[0] < 2:
    print(">>> α < 2: хвосты степенные, дисперсия формально бесконечна.")

In [ ]:
# Визуальное сравнение подобранных распределений
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_grid = np.linspace(log_ret.min(), log_ret.max(), 500)

# Плотности на фоне гистограммы
axes[0].hist(r, bins=100, density=True, alpha=0.5, color="grey", label="Эмпирическое")
axes[0].plot(x_grid, st.norm.pdf(x_grid, *norm_params), "r-",  lw=2, label="Нормальное")
axes[0].plot(x_grid, st.t.pdf(x_grid, *t_params),      "b--", lw=2, label=f"Стьюдента (ν={t_params[0]:.1f})")
axes[0].plot(x_grid, st.levy_stable.pdf(x_grid, *stable_params), "g-.", lw=2,
             label=f"α-stable (α={stable_params[0]:.2f})")
axes[0].set_title("Сравнение подобранных распределений")
axes[0].set_xlabel("Лог-доходность")
axes[0].set_ylabel("Плотность")
axes[0].legend()

# QQ-plot против распределения Стьюдента
st.probplot(r, dist=st.t, sparams=(t_params[0],), plot=axes[1])
axes[1].set_title(f"QQ-plot: лог-доходности vs t(ν={t_params[0]:.1f})")
axes[1].get_lines()[0].set_markersize(2)

plt.tight_layout()
plt.show()

In [ ]:
# Log-log график хвостов: P(|r| > x) vs x
# Сравнение скорости убывания хвостов: нормальное (экспоненциальное) vs степенное
abs_r = np.sort(np.abs(r))[::-1]
empirical_cdf = np.arange(1, len(abs_r)+1) / len(abs_r)   # P(|r| > x)

x_tail = np.linspace(abs_r[-1], abs_r[0], 500)
norm_tail = 2 * (1 - st.norm.cdf(x_tail, 0, norm_params[1]))
t_tail = 2 * (1 - st.t.cdf(x_tail, *t_params[:1], 0, t_params[2]))

plt.figure(figsize=(10, 6))
plt.loglog(abs_r, empirical_cdf, "k.", markersize=1, label="Эмпирический хвост")
plt.loglog(x_tail, norm_tail, "r-", lw=2, label="Нормальное")
plt.loglog(x_tail, t_tail, "b--", lw=2, label=f"Стьюдента (ν={t_params[0]:.1f})")
plt.xlabel("|r|")
plt.ylabel("P(|r| > x)")
plt.title("Убывание хвостов распределения (log-log)")
plt.legend()
plt.show()

## 4. Анализ стационарности

Стационарность — необходимое условие для корректного применения ML к временному ряду. Нестационарные признаки могут давать ложные корреляции.

### Тест Дики-Фуллера (ADF)
Проверяет $H_0$: ряд имеет единичный корень (нестационарен).
Модель: $\Delta X_t = \alpha + \gamma X_{t-1} + \sum_{i=1}^{p} \delta_i \Delta X_{t-i} + \varepsilon_t$.
$H_0\!: \gamma = 0$ (единичный корень), $H_1\!: \gamma < 0$ (стационарность).

### Тест KPSS
Комплементарный тест. $H_0$: ряд стационарен. $H_1$: нестационарен.

| ADF | KPSS | Вывод |
|-----|------|-------|
| Отвергнута | Не отвергнута | **Стационарный** |
| Не отвергнута | Отвергнута | **Нестационарный** |
| Отвергнута | Отвергнута | Тренд-стационарный |
| Не отвергнута | Не отвергнута | Требуется дополнительный анализ |

In [ ]:
def stationarity_tests(series, name):
    """Запускает ADF и KPSS тесты, возвращает сводку."""
    # ADF: H0 = единичный корень (нестационарен)
    adf_stat, adf_p, adf_lags, _, adf_crit, _ = adfuller(series, autolag="AIC")

    # KPSS: H0 = стационарен
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series, regression="c", nlags="auto")

    alpha = 0.05
    adf_reject = adf_p < alpha
    kpss_reject = kpss_p < alpha

    if adf_reject and not kpss_reject:
        verdict = "Стационарный"
    elif not adf_reject and kpss_reject:
        verdict = "Нестационарный"
    elif adf_reject and kpss_reject:
        verdict = "Тренд-стационарный"
    else:
        verdict = "Неопределённо"

    return {
        "Ряд": name,
        "ADF стат.": adf_stat, "ADF p": adf_p, "ADF отверг.": adf_reject,
        "KPSS стат.": kpss_stat, "KPSS p": kpss_p, "KPSS отверг.": kpss_reject,
        "Вывод": verdict,
    }

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # KPSS выдаёт InterpolationWarning
    rows = [
        stationarity_tests(close, "BTC цена (close)"),
        stationarity_tests(log_ret, "BTC лог-доходность"),
        stationarity_tests(np.log(close), "BTC ln(цена)"),
    ]

stat_df = pd.DataFrame(rows)
display(stat_df[["Ряд", "ADF стат.", "ADF p", "KPSS стат.", "KPSS p", "Вывод"]])

In [ ]:
# Визуальная демонстрация стационарности: скользящее среднее и std
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Цена — нестационарна
axes[0, 0].plot(close, lw=0.5)
axes[0, 0].plot(close.rolling(252).mean(), "r-", lw=2, label="MA-252")
axes[0, 0].set_title("BTC цена закрытия")
axes[0, 0].legend()

axes[0, 1].plot(close.rolling(252).std(), "r-", lw=1)
axes[0, 1].set_title("Скользящее ст. откл. цены (252д)")

# Лог-доходности — стационарны
axes[1, 0].plot(log_ret, lw=0.3, alpha=0.7)
axes[1, 0].plot(log_ret.rolling(252).mean(), "r-", lw=2, label="MA-252")
axes[1, 0].set_title("BTC лог-доходности")
axes[1, 0].legend()

axes[1, 1].plot(log_ret.rolling(252).std(), "r-", lw=1)
axes[1, 1].set_title("Скользящее ст. откл. доходностей (252д)")

plt.tight_layout()
plt.show()

print(">>> Цена — нестационарна (среднее и дисперсия растут со временем).")
print(">>> Доходности — стационарны (среднее ≈ const, дисперсия колеблется, но не растёт).")
print(">>> Вывод: в качестве признаков использовать доходности (returns), а не цены (prices).")

## 5. Автокорреляция и кластеризация волатильности

Автокорреляционная функция (ACF): $\rho(k) = \frac{\text{Cov}(r_t, r_{t-k})}{\text{Var}(r_t)}$.

**Стилизованные факты финансовых рядов:**
1. Доходности $r_t$ почти не имеют автокорреляции (слабая форма эффективности рынка).
2. Квадраты доходностей $r_t^2$ и $|r_t|$ имеют **сильную, медленно убывающую** автокорреляцию → **кластеризация волатильности**: за периодами высокой волатильности следуют периоды высокой волатильности.

### Тест Льюнга-Бокса
$Q(h) = n(n+2)\sum_{k=1}^{h} \frac{\hat\rho_k^2}{n-k} \sim \chi^2(h)$.
$H_0$: первые $h$ автокорреляций равны нулю.

### Тест ARCH Энгла
Проверяет наличие ARCH-эффектов (авторегрессионной условной гетероскедастичности):
$\varepsilon_t^2 = \alpha_0 + \alpha_1\varepsilon_{t-1}^2 + \ldots + \alpha_p\varepsilon_{t-p}^2 + u_t$.
$H_0$: $\alpha_1 = \ldots = \alpha_p = 0$ (нет ARCH-эффектов).

In [ ]:
# ACF/PACF доходностей и квадратов доходностей
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(log_ret,      lags=40, ax=axes[0, 0], title="ACF лог-доходностей $r_t$")
plot_pacf(log_ret,     lags=40, ax=axes[0, 1], title="PACF лог-доходностей $r_t$")
plot_acf(log_ret**2,   lags=40, ax=axes[1, 0], title="ACF квадратов доходностей $r_t^2$")
plot_acf(log_ret.abs(), lags=40, ax=axes[1, 1], title="ACF |$r_t$|")

plt.tight_layout()
plt.show()

print("Верхний ряд: ACF доходностей — ожидаем слабую/нулевую автокорреляцию.")
print("Нижний ряд: ACF r² и |r| — ожидаем сильную, медленно убывающую (кластеризация волатильности).")

In [ ]:
# Тест Льюнга-Бокса на доходности и квадраты
lags_test = [5, 10, 20]

lb_ret = acorr_ljungbox(log_ret, lags=lags_test, return_df=True)
lb_ret.index = [f"lag={l}" for l in lags_test]
lb_ret.columns = ["Q-стат. (r_t)", "p-value (r_t)"]

lb_sq = acorr_ljungbox(log_ret**2, lags=lags_test, return_df=True)
lb_sq.index = [f"lag={l}" for l in lags_test]
lb_sq.columns = ["Q-стат. (r_t²)", "p-value (r_t²)"]

lb_all = pd.concat([lb_ret, lb_sq], axis=1)
display(lb_all)

# Тест ARCH Энгла
arch_res = het_arch(log_ret, nlags=10)
print(f"\nТест ARCH Энгла (10 лагов):")
print(f"  LM-статистика = {arch_res[0]:.2f}")
print(f"  p-value = {arch_res[1]:.2e}")
if arch_res[1] < 0.05:
    print("  >>> ARCH-эффекты присутствуют: волатильность кластеризуется.")
    print("  >>> Это обосновывает включение скользящей волатильности в качестве признака.")

## 6. Экспонента Хёрста (R/S-анализ)

Экспонента Хёрста $H$ характеризует долговременную память временного ряда.

### Метод R/S (Rescaled Range)
Для подсерии длины $n$:
1. Вычисляем среднее $\bar{r}$ и кумулятивное отклонение $Y_t = \sum_{i=1}^{t}(r_i - \bar{r})$.
2. Размах $R = \max(Y) - \min(Y)$, стандартное отклонение $S$.
3. Нормированный размах $R/S$.

Закон Хёрста: $\mathbb{E}[R/S] \sim c \cdot n^H$ при $n \to \infty$.

Оценка $H$ — наклон регрессии $\ln(R/S)$ на $\ln(n)$.

### Интерпретация
| $H$ | Поведение |
|-----|-----------|
| $H = 0.5$ | Случайное блуждание (нет памяти) |
| $H > 0.5$ | **Персистентность** (тренды продолжаются) — оправдывает momentum-признаки |
| $H < 0.5$ | **Антиперсистентность** (mean-reversion) — оправдывает mean-reversion-признаки |

Связь с дробным броуновским движением $B_H(t)$: $H$ определяет автоковариационную структуру приращений.

In [ ]:
def rs_hurst(series):
    """
    Оценка экспоненты Хёрста методом R/S (Rescaled Range).

    Для каждого масштаба n вычисляем среднее R/S по неперекрывающимся блокам,
    затем оцениваем H как наклон log(R/S) vs log(n).
    """
    x = np.asarray(series, dtype=float)
    N = len(x)

    # Выбираем масштабы n (от 10 до N//2, логарифмически)
    min_n, max_n = 10, N // 2
    n_values = np.unique(np.logspace(np.log10(min_n), np.log10(max_n), 30).astype(int))

    rs_means = []
    for n in n_values:
        n_blocks = N // n
        if n_blocks < 1:
            continue
        rs_list = []
        for b in range(n_blocks):
            block = x[b*n : (b+1)*n]
            mean_b = block.mean()
            # Кумулятивное отклонение от среднего
            cumdev = np.cumsum(block - mean_b)
            R = cumdev.max() - cumdev.min()   # размах
            S = block.std(ddof=1)             # стандартное отклонение
            if S > 0:
                rs_list.append(R / S)
        if rs_list:
            rs_means.append((n, np.mean(rs_list)))

    ns, rs = zip(*rs_means)
    log_n  = np.log(ns)
    log_rs = np.log(rs)

    # Линейная регрессия: log(R/S) = H * log(n) + c
    H, c = np.polyfit(log_n, log_rs, 1)
    return H, np.array(ns), np.array(rs)

# Оценка H для всего ряда лог-доходностей
H_global, ns, rs = rs_hurst(log_ret.values)
print(f"Экспонента Хёрста (R/S): H = {H_global:.4f}")

if H_global > 0.55:
    print(">>> H > 0.5: ряд персистентен (тренды продолжаются).")
    print(">>> Обоснование momentum-признаков (MA, returns).")
elif H_global < 0.45:
    print(">>> H < 0.5: ряд антиперсистентен (mean-reversion).")
    print(">>> Обоснование mean-reversion-признаков (dist_to_ma, z-score).")
else:
    print(">>> H ≈ 0.5: близко к случайному блужданию.")

In [ ]:
# График R/S-анализа: log(R/S) vs log(n) и регрессионная прямая
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

log_n  = np.log(ns)
log_rs = np.log(rs)
H_fit, c_fit = np.polyfit(log_n, log_rs, 1)

axes[0].scatter(log_n, log_rs, s=20, label="Эмпирические R/S")
axes[0].plot(log_n, H_fit * log_n + c_fit, "r-", lw=2,
             label=f"Регрессия: H = {H_fit:.3f}")
axes[0].plot(log_n, 0.5 * log_n + c_fit, "k--", alpha=0.5,
             label="H = 0.5 (случайное блуждание)")
axes[0].set_xlabel("ln(n)")
axes[0].set_ylabel("ln(R/S)")
axes[0].set_title("R/S-анализ: оценка экспоненты Хёрста")
axes[0].legend()

# Скользящий Hurst (окно 252 дня)
rolling_H = log_ret.rolling(window=252).apply(lambda w: rs_hurst(w.values)[0], raw=False)
axes[1].plot(rolling_H, lw=0.8)
axes[1].axhline(0.5, color="red", ls="--", lw=1.5, label="H = 0.5")
axes[1].set_title("Скользящая экспонента Хёрста (окно 252 дня)")
axes[1].set_ylabel("H")
axes[1].legend()

plt.tight_layout()
plt.show()

print(">>> Скользящий H показывает, что рыночный режим меняется со временем.")
print(">>> Это обосновывает включение скользящего H как признака в ML-модель.")

## 7. Выводы для моделирования

Сводка статистических свойств доходностей BTC и их следствия для построения ML-модели.

In [ ]:
# Сводная таблица результатов анализа
summary = pd.DataFrame({
    "Свойство": [
        "Нормальность",
        "Тяжёлые хвосты",
        "Стационарность цены",
        "Стационарность доходностей",
        "Автокорреляция доходностей",
        "Кластеризация волатильности",
        "Экспонента Хёрста (глобальная)",
    ],
    "Результат": [
        "Отвергнута (JB, SW, KS)",
        f"Стьюдент ν≈{t_params[0]:.1f}, α-stable α≈{stable_params[0]:.2f}",
        "Нестационарна (ADF не отвергнут, KPSS отвергнут)",
        "Стационарны (ADF отвергнут, KPSS не отвергнут)",
        "Слабая/отсутствует",
        f"Присутствует (ARCH p={arch_res[1]:.2e})",
        f"H = {H_global:.3f}",
    ],
    "Следствие для модели": [
        "Не использовать методы, предполагающие нормальность. Древесные модели подходят.",
        "Экстремальные движения чаще ожидаемого. Робастное масштабирование вместо z-score.",
        "НЕ использовать цену как признак напрямую.",
        "Использовать доходности (returns) как признаки.",
        "Прошлые доходности слабо предсказывают будущие напрямую.",
        "Включить скользящую волатильность (btc_vol_20) как признак.",
        "Включить скользящий H как признак; режим меняется со временем.",
    ],
})
display(summary)

### Итоговые рекомендации для признаков и модели

На основе проведённого анализа:

1. **Признаки на основе доходностей** (ret_1d, ret_7d, ret_30d) — обоснованы стационарностью. Сырая цена не должна использоваться как признак.

2. **Скользящая волатильность** (btc_vol_20) — обоснована наличием ARCH-эффектов и кластеризацией волатильности.

3. **Скользящая экспонента Хёрста** — отражает текущий рыночный режим (тренд vs mean-reversion). Динамический характер H обосновывает его как признак.

4. **Древесные модели** (Random Forest, XGBoost) — предпочтительны, т.к. не предполагают нормальность распределения признаков и робастны к выбросам.

5. **Дополнительные признаки** для включения: RSI, MACD (momentum), ширина полос Боллинджера (волатильность), Fear & Greed Index (настроение), Federal Funds Rate (макро).